# L12 · Prompts dinámicos con Jinja sobre un ticket de ACME

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

Retomamos el ticket de L1: el cliente C-1042 de **ACME** reporta dos cargos iguales en su tarjeta débito. Ahora cambiamos la forma de construir el prompt. En L1 las plantillas tenían huecos fijos (`{context}`, `{question}`). Aquí el prompt **cambia según los datos**: si el ticket trae la fecha de los cargos, cuántos cargos incluye o qué reglas aplicar. Para eso usamos **Jinja**.

## Objetivos

Al terminar podrás:

1. Escribir plantillas Jinja con variables `{{ }}`, condicionales `{% if %}` y bucles `{% for %}`.
2. Guardar la plantilla en un archivo y detectar variables faltantes con `StrictUndefined`.
3. Separar plantilla, contexto y mensaje del usuario.
4. Validar la respuesta con un esquema Pydantic.
5. Comparar dos variantes del mismo prompt y ubicar sus trazas en LangSmith.

## Metodología

- **Un concepto por sección**, igual que en L1. Las secciones 1–6 solo construyen texto: **no llaman a Claude**.
- **Mirar el prompt antes de enviarlo.** Imprimimos cada prompt renderizado; es lo que recibirá el modelo.
- **Mismo ticket y mismo modelo** en todo el recorrido.

## Contenido

- [0. Preparación del modelo y las trazas](#0.-Preparación-del-modelo-y-las-trazas)
- [1. Repaso: la plantilla de L1](#1.-Repaso:-la-plantilla-de-L1)
- [2. Fijar un valor con `partial_variables`](#2.-Fijar-un-valor-con-partial_variables)
- [3. Primera plantilla Jinja](#3.-Primera-plantilla-Jinja)
- [4. Condicionales con `{% if %}`](#4.-Condicionales-con-{%-if-%})
- [5. Bucles con `{% for %}`](#5.-Bucles-con-{%-for-%})
- [6. Plantilla en archivo y `StrictUndefined`](#6.-Plantilla-en-archivo-y-StrictUndefined)
- [7. Separar plantilla, contexto y mensaje del usuario](#7.-Separar-plantilla,-contexto-y-mensaje-del-usuario)
- [8. Validar la salida con Pydantic](#8.-Validar-la-salida-con-Pydantic)
- [9. Comparar dos variantes y sus trazas](#9.-Comparar-dos-variantes-y-sus-trazas)
- [Práctica breve](#Práctica-breve)
- [Referencias](#Referencias)

**Preparación:** igual que en L1. Ver [SETUP.md](../SETUP.md). Las trazas de LangSmith son opcionales (sección 0).

Todo el recorrido realiza **4 llamadas a Claude Haiku**: una en la sección 7, una en la 8 y dos en la 9. Repetir celdas consume tokens.

## 0. Preparación del modelo y las trazas

El modelo es el mismo de L1. **LangSmith** registra cada llamada como una *traza*: el prompt enviado, la respuesta, los tokens y la latencia. Se activa solo con variables de entorno; el código no cambia.

Para activarlo, en tu `.env`:

```
LANGSMITH_API_KEY=<tu clave>
LANGSMITH_PROJECT=advancedgenai
LANGSMITH_TRACING=true
```

Sin clave, el notebook funciona igual y no envía trazas. La capa gratuita tiene un límite mensual de trazas: activa el tracing solo cuando lo necesites. Usa solo datos sintéticos cuando las trazas estén activas.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=400,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)
print("Trazas en LangSmith:", os.getenv("LANGSMITH_TRACING", "false"))

## 1. Repaso: la plantilla de L1

En L1, `ChatPromptTemplate` rellenaba huecos con nombre: `{context}` y `{question}`. Esta celda **no llama al modelo**: solo construye los mensajes.

En L1 usamos `invoke`. Aquí usamos `format_messages`, que devuelve directamente la lista de mensajes. El resultado es el mismo, pero con LangSmith activo `invoke` registra una traza más por cada plantilla. Así ahorramos trazas.

La plantilla siempre produce el mismo texto con otros valores. No puede decir «si el ticket no trae la fecha, avisa» ni repetir una línea por cada cargo. Eso es lo que añade Jinja.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ticket_text = (
    "Ticket T-2048: el cliente C-1042 de ACME reporta dos cargos iguales de "
    "185.000 COP en su tarjeta débito, hechos el 12 de octubre en la misma tienda en línea."
)

SYSTEM_PROMPT = """\
Responde brevemente en español usando solo el contexto.
Si falta el dato, responde 'No se indica en el ticket'.
"""

chat_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Contexto: {context}"),
    ("human", "Pregunta: {question}"),
])
chat_prompt = chat_template.format_messages(
    context=ticket_text,
    question="¿Cuál es el código de autorización de los cargos?",
)
for message in chat_prompt:
    message.pretty_print()

## 2. Fijar un valor con `partial_variables`

Algunos valores no los escribe el usuario, como la fecha de hoy. `partial_variables` los fija al crear la plantilla; después solo pasamos lo que cambia.

In [ ]:
from datetime import date
from langchain_core.prompts import PromptTemplate

today = date.today().isoformat()

dated_template = PromptTemplate.from_template(
    "Hoy es {today}.\nResume este ticket en una frase: {ticket}",
    partial_variables={"today": today},
)
print(dated_template.format(ticket=ticket_text))

## 3. Primera plantilla Jinja

Con `template_format="jinja2"`, LangChain interpreta la plantilla con Jinja. Las variables se escriben con **doble llave**: `{{ variable }}`.

La primera plantilla tiene un error frecuente: `{ user_message }` con **una** llave. Jinja no la reconoce como variable y la deja como texto. Observa la salida: el mensaje del usuario nunca aparece.

In [ ]:
broken_template = PromptTemplate.from_template(
    "Revisa el ticket.\nMensaje del cliente: { user_message }",
    template_format="jinja2",
)
print(broken_template.input_variables)
print(broken_template.format())

Con `{{ user_message }}`, LangChain detecta la variable y la exige al renderizar.

In [ ]:
JINJA_TEMPLATE = """\
Revisa el ticket.
Mensaje del cliente: {{ user_message }}
"""
jinja_template = PromptTemplate.from_template(
    JINJA_TEMPLATE,
    template_format="jinja2",
)
print(jinja_template.input_variables)
print(jinja_template.format(user_message="¿Por qué me cobraron dos veces?"))

## 4. Condicionales con `{% if %}`

Un ticket puede traer la fecha de los cargos o no. Con `{% if %}` el prompt incluye la fecha cuando existe y, si no, una instrucción para no inventarla. `{% else %}` y `{% endif %}` cierran el bloque.

Las variables del `if` también son obligatorias en LangChain: un campo opcional se pasa **explícitamente** como `None`. Renderizamos los dos casos.

In [ ]:
CONDITIONAL_TEMPLATE = """\
Ticket: {{ ticket }}
{% if charge_date %}\
Fecha de los cargos: {{ charge_date }}\
{% else %}\
Fecha de los cargos: no se indica. No la inventes.\
{% endif %}
"""
date_template = PromptTemplate.from_template(
    CONDITIONAL_TEMPLATE,
    template_format="jinja2",
)
print(date_template.format(ticket=ticket_text, charge_date="2026-10-12"))
print("----" * 18)
print(date_template.format(ticket=ticket_text, charge_date=None))

## 5. Bucles con `{% for %}`

Las reglas de revisión cambian según el producto o el tipo de reclamo. Con `{% for %}` escribimos una línea por regla, sin tocar la plantilla. `loop.index` numera desde 1.

Es la misma idea del prompt de clasificación de Grab en *AI Engineering*: una lista de definiciones que el modelo debe aplicar.

In [ ]:
rules = [
    "Usa solo los datos del ticket; si falta un dato, inclúyelo en los datos faltantes.",
    "Si la fecha de los cargos es relativa (por ejemplo, 'ayer'), márcala como ambigua y no la conviertas en una fecha.",
    "No conviertas montos entre monedas ni redondees los valores.",
]

LOOP_TEMPLATE = """\
Reglas de revisión:
{% for rule in rules %}
{{ loop.index }}. {{ rule }}\
{% endfor %}
"""
rules_template = PromptTemplate.from_template(
    LOOP_TEMPLATE,
    template_format="jinja2",
)
print(rules_template.format(rules=rules))

## 6. Plantilla en archivo y `StrictUndefined`

Una plantilla real crece; conviene guardarla en el repositorio y revisarla como el código. Abre [`datasets/acme_ticket.jinja`](../datasets/acme_ticket.jinja): reúne la fecha de hoy, el ticket, el `for` de cargos, el `if` de la fecha y el `for` de reglas.

Para leerla usamos Jinja directamente:

- `FileSystemLoader("../datasets")` busca la plantilla en la carpeta `datasets/` del repositorio.
- `StrictUndefined` **falla** si falta una variable, en lugar de dejar un hueco.
- `trim_blocks` y `lstrip_blocks` quitan las líneas en blanco que dejan los bloques `{% %}`.

El ticket ahora es un diccionario: separamos los datos (contexto) de la plantilla. Renderizamos dos contextos: con fecha exacta y con fecha relativa.

In [ ]:
from jinja2 import Environment, FileSystemLoader, StrictUndefined

env = Environment(
    loader=FileSystemLoader("../datasets"),
    undefined=StrictUndefined,
    trim_blocks=True,
    lstrip_blocks=True,
)
ticket_template = env.get_template("acme_ticket.jinja")

ticket = {
    "code": "T-2048",
    "customer": "C-1042",
    "product": "tarjeta débito",
    "charges": [
        {"amount": 185000, "currency": "COP", "merchant": "tienda en línea"},
        {"amount": 185000, "currency": "COP", "merchant": "tienda en línea"},
    ],
    "charge_date": "2026-10-12",
}
print(ticket_template.render(today=today, ticket=ticket, rules=[]))

El segundo ticket trae tres cargos, uno de ellos en otra moneda, y una fecha relativa. Pasamos las reglas: el `for` las numera al final del prompt.

In [ ]:
ambiguous_ticket = {
    "code": "T-2049",
    "customer": "C-1042",
    "product": "tarjeta débito",
    "charges": [
        {"amount": 185000, "currency": "COP", "merchant": "tienda en línea"},
        {"amount": 185000, "currency": "COP", "merchant": "tienda en línea"},
        {"amount": 12, "currency": "USD", "merchant": "suscripción de streaming"},
    ],
    "charge_date": "ayer",
}
print(ticket_template.render(today=today, ticket=ambiguous_ticket, rules=rules))

### 6.1 Detectar una variable faltante

Quitamos el cliente del ticket. Con el `Environment` por defecto, Jinja deja `Cliente:` vacío **sin avisar**. Con `StrictUndefined` obtenemos un error que nombra el dato faltante, antes de gastar una llamada.

In [ ]:
from jinja2 import UndefinedError

ticket_without_customer = {key: value for key, value in ticket.items() if key != "customer"}

lenient_env = Environment(loader=FileSystemLoader("../datasets"), trim_blocks=True, lstrip_blocks=True)
print(lenient_env.get_template("acme_ticket.jinja").render(
    today=today, ticket=ticket_without_customer, rules=[]
))

try:
    ticket_template.render(today=today, ticket=ticket_without_customer, rules=[])
except UndefinedError as error:
    print("StrictUndefined:", error)

## 7. Separar plantilla, contexto y mensaje del usuario

Ahora armamos los mensajes para Claude:

- **Plantilla:** `acme_ticket.jinja`, versionada en el repositorio.
- **Contexto:** el ticket y las reglas, que renderizamos como mensaje de sistema.
- **Mensaje del usuario:** lo que escribe el cliente. Entra como **dato**, nunca como parte de la plantilla.

Por eso el mensaje del usuario no pasa por Jinja. Si alguien escribe `{{ ticket.customer }}`, debe llegar como texto literal. Una plantilla Jinja solo debe venir de una fuente confiable, nunca del usuario.

In [ ]:
review_template = ChatPromptTemplate.from_messages([
    ("system", "{system_prompt}"),
    ("human", "{user_message}"),
])

system_prompt = ticket_template.render(today=today, ticket=ambiguous_ticket, rules=rules)
messages = review_template.format_messages(
    system_prompt=system_prompt,
    user_message="¿Qué falta para abrir la disputa de los cargos? {{ ticket.customer }}",
)
print(messages[-1].content)

El mensaje humano conserva `{{ ticket.customer }}` como texto. Enviamos los mensajes al modelo.

In [ ]:
response = model.invoke(messages)
for message in messages + [response]:
    message.pretty_print()

## 8. Validar la salida con Pydantic

Como en L1, `with_structured_output` pide una respuesta con forma. `TicketReview` describe lo que queremos de la revisión. `charge_date` es opcional: `None` cuando el ticket no trae una fecha exacta. `issues` reúne los datos faltantes o ambiguos.

La estructura garantiza los tipos, no que el contenido sea correcto: revisa la salida contra el ticket.

In [ ]:
from pydantic import BaseModel, Field

class TicketReview(BaseModel):
    """Revisión de un ticket antes de abrir una disputa de cargos."""
    customer: str = Field(description="Código del cliente que reporta el problema.")
    disputed_charges: int = Field(description="Número de cargos que el cliente pide revisar.")
    charge_date: str | None = Field(
        description="Fecha exacta AAAA-MM-DD; None si no hay fecha exacta."
    )
    issues: list[str] = Field(description="Datos faltantes o ambiguos.")

review_model = model.with_structured_output(TicketReview)
review = review_model.invoke(messages)
print(review.model_dump_json(indent=2))

### 8.1 Validar sin llamar al modelo

Pydantic también valida datos que llegan de otro lugar. Aquí `disputed_charges` es texto y falta `issues`: el error dice qué campo falla y por qué. Esta celda **no llama a Claude**.

In [ ]:
from pydantic import ValidationError

try:
    TicketReview.model_validate(
        {"customer": "C-1042", "disputed_charges": "dos", "charge_date": None}
    )
except ValidationError as error:
    print(error)

## 9. Comparar dos variantes y sus trazas

Cambiar un prompt es un experimento: hay que comparar sobre el mismo ticket. Renderizamos **dos variantes** con la misma plantilla:

- **Variante A:** sin reglas (`rules=[]`).
- **Variante B:** con las tres reglas de la sección 5.

El argumento `config` nombra cada llamada (`run_name`) y le agrega etiquetas y metadatos. Si las trazas están activas, así las encuentras en LangSmith.

In [ ]:
variants = {"A": [], "B": rules}
variant_reviews = {}

for name, variant_rules in variants.items():
    variant_messages = review_template.format_messages(
        system_prompt=ticket_template.render(
            today=today, ticket=ambiguous_ticket, rules=variant_rules
        ),
        user_message="Revisa el ticket antes de abrir la disputa.",
    )
    variant_reviews[name] = review_model.invoke(
        variant_messages,
        config={
            "run_name": f"L12 variante {name}",
            "tags": ["l12", f"variante-{name.lower()}"],
            "metadata": {"template": "acme_ticket.jinja", "rules": len(variant_rules)},
        },
    )

for name, variant_review in variant_reviews.items():
    print(f"Variante {name}:", variant_review.model_dump_json(indent=2))

Compara las dos salidas campo por campo, empezando por `charge_date`. Sin reglas, la variante A puede calcular «ayer» a partir de la fecha de hoy y devolver una fecha que el cliente nunca dio. La regla 2 de B pide marcarla como ambigua y dejar `None`. Mira también `disputed_charges`: ¿alguna variante cuenta el cargo en dólares como parte del reclamo, aunque el cliente no dijo nada de él? Ninguna regla habla de eso todavía; sería la siguiente regla que podrías añadir. Si las salidas coinciden, B paga más tokens de entrada sin mejorar la revisión.

**En LangSmith** (si activaste las trazas): abre el proyecto `advancedgenai` y filtra por la etiqueta `l12`. Abre `L12 variante A` y `L12 variante B`: compara el mensaje de sistema que recibió el modelo, la salida estructurada, los tokens y la latencia. La variante B usa más tokens de entrada; decide si la mejora lo justifica.

Dos ejecuciones no bastan para elegir una variante. En la semana de evaluación compararemos variantes sobre un dataset con evaluadores.

## Práctica breve

1. Añade una regla a `rules`, por ejemplo sobre cargos en otra moneda, y vuelve a renderizar la sección 6. Comprueba que el `for` la numera.
2. Escribe en `datasets/` una plantilla para otro tipo de ticket de ACME, como un bloqueo de tarjeta, con un `if` y un `for`.
3. Quita `today` de un `render` y observa el error de `StrictUndefined`.
4. Escribe una variante C de `acme_ticket.jinja` en otro archivo de `datasets/`, cámbiala en una sola instrucción y compárala con B usando un nuevo `run_name`. Explica qué cambió en la salida.

**Ampliación opcional:** `review_model.batch([...])` revisa varios tickets en una sola instrucción, y `model.stream(messages)` muestra la respuesta de la sección 7 por fragmentos (L1, sección 10).

## Referencias

**Learning LangChain**, Mayo Oshin y Nuno Campos, O'Reilly: [scripts del capítulo 1](https://github.com/langchain-ai/learning-langchain/tree/master/ch1/py) (`f-chat-prompt.py`, `g-chat-prompt-model.py`, `h-structured.py` y `j-methods.py`). El libro no cubre Jinja. El ejemplo de reglas sigue la idea del prompt de Grab en [*AI Engineering*, prompt-examples.md](https://github.com/chiphuyen/aie-book/blob/f375e0065ef19e4f411b809a2f9a2d65582db287/prompt-examples.md).

- [Plantillas Jinja](https://jinja.palletsprojects.com/en/stable/templates/) y [`StrictUndefined`](https://jinja.palletsprojects.com/en/stable/api/#jinja2.StrictUndefined)
- [Trazas en LangSmith](https://docs.langchain.com/langsmith/observability-quickstart)